### Text To Sql Agent

In [1]:
from langchain_community.utilities import SQLDatabase
from sqlalchemy import create_engine
import os 
from dotenv import load_dotenv

C:\Users\HP\AppData\Local\Temp\ipykernel_4796\4288095332.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [2]:
load_dotenv()

True

In [3]:
DB_USER=os.getenv("DB_USER")
DB_NAME=os.getenv("DB_NAME")
DB_PASSWORD=os.getenv("DB_PASSWORD")
DB_PORT=os.getenv("DB_PORT")
DB_HOST=os.getenv("DB_HOST")

In [4]:
from sqlalchemy.engine import URL

connection_url=URL.create(
    "mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

In [5]:
engine=create_engine(
    connection_url
)

In [6]:
db=SQLDatabase(
    engine,
    sample_rows_in_table_info=0,
    ignore_tables=["_demodb_import"]
)

In [7]:
def get_schema():
    return db.get_table_info()

In [8]:
schema=get_schema()

In [9]:
print(schema)


CREATE TABLE album (
	`AlbumId` BIGINT NOT NULL AUTO_INCREMENT, 
	`Title` VARCHAR(160) NOT NULL, 
	`ArtistId` BIGINT NOT NULL, 
	PRIMARY KEY (`AlbumId`), 
	CONSTRAINT album_ibfk_1 FOREIGN KEY(`ArtistId`) REFERENCES artist (`ArtistId`)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci


CREATE TABLE artist (
	`ArtistId` BIGINT NOT NULL AUTO_INCREMENT, 
	`Name` VARCHAR(120), 
	PRIMARY KEY (`ArtistId`)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci


CREATE TABLE customer (
	`CustomerId` BIGINT NOT NULL AUTO_INCREMENT, 
	`FirstName` VARCHAR(40) NOT NULL, 
	`LastName` VARCHAR(20) NOT NULL, 
	`Company` VARCHAR(80), 
	`Address` VARCHAR(70), 
	`City` VARCHAR(40), 
	`State` VARCHAR(40), 
	`Country` VARCHAR(40), 
	`PostalCode` VARCHAR(10), 
	`Phone` VARCHAR(24), 
	`Fax` VARCHAR(24), 
	`Email` VARCHAR(60) NOT NULL, 
	`SupportRepId` BIGINT, 
	PRIMARY KEY (`CustomerId`), 
	CONSTRAINT customer_ibfk_1 FOREIGN KEY(`SupportRepId`) REFERENCES employee (`EmployeeId`)


In [10]:
def run_query(query):
    print(f"Query being executed:{query}")
    return db.run(query)

In [11]:
run_query("""
SELECT Artist.Name
FROM Artist
LIMIT 5;
""")

Query being executed:
SELECT Artist.Name
FROM Artist
LIMIT 5;



"[('AC/DC',), ('Accept',), ('Aerosmith',), ('Alanis Morissette',), ('Alice In Chains',)]"

### LLM Initilization

In [12]:
from langchain_groq import ChatGroq

GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [13]:
llm=ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=GROQ_API_KEY
)

### SQL Generator

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def write_sql_query(llm):

    template="""
        Based on the table schema below , write a SQL Query
        that would answer the user's question.

        Schema:{schema}

        Question:{question}

        SQL Query:
    """

    prompt=ChatPromptTemplate.from_messages([
        ("system","""
            You are an expert SQL Query generator.
            Generate only valid MySQL SQL Queries.
            Do not include explainations or markdowns.
        """),
        ("human",template)
    ])

    return prompt|llm|StrOutputParser()

In [15]:
# Test generation

sql_chain=write_sql_query(llm)

response=sql_chain.invoke({
    "schema":get_schema(),
    "question":"Show all tracks by AC/DC"
})

print(response)

SELECT t.* 
FROM track t
JOIN album a ON t.AlbumId = a.AlbumId
JOIN artist ar ON a.ArtistId = ar.ArtistId
WHERE ar.Name = 'AC/DC';


In [16]:
result=run_query(response)

print(result)

Query being executed:SELECT t.* 
FROM track t
JOIN album a ON t.AlbumId = a.AlbumId
JOIN artist ar ON a.ArtistId = ar.ArtistId
WHERE ar.Name = 'AC/DC';
[(1, 'For Those About To Rock (We Salute You)', 1, 1, 1, 'Angus Young, Malcolm Young, Brian Johnson', 343719, 11170334, Decimal('0.99')), (6, 'Put The Finger On You', 1, 1, 1, 'Angus Young, Malcolm Young, Brian Johnson', 205662, 6713451, Decimal('0.99')), (7, "Let's Get It Up", 1, 1, 1, 'Angus Young, Malcolm Young, Brian Johnson', 233926, 7636561, Decimal('0.99')), (8, 'Inject The Venom', 1, 1, 1, 'Angus Young, Malcolm Young, Brian Johnson', 210834, 6852860, Decimal('0.99')), (9, 'Snowballed', 1, 1, 1, 'Angus Young, Malcolm Young, Brian Johnson', 203102, 6599424, Decimal('0.99')), (10, 'Evil Walks', 1, 1, 1, 'Angus Young, Malcolm Young, Brian Johnson', 263497, 8611245, Decimal('0.99')), (11, 'C.O.D.', 1, 1, 1, 'Angus Young, Malcolm Young, Brian Johnson', 199836, 6566314, Decimal('0.99')), (12, 'Breaking The Rules', 1, 1, 1, 'Angus Young

### SQL Validator

In [17]:
import sqlglot
from sqlglot import exp

In [18]:
def validate_sql(query):

    try:
        parsed=sqlglot.parse(query,read="mysql")

        if len(parsed)!=1:
            return {
                "valid":False,
                "operation":None,
                "message":"Only one SQL Statement is allowed"
            }

        statement=parsed[0]

        if isinstance(statement,exp.Select):
            operation="SELECT"

        elif isinstance(statement,exp.Insert):
            operation="INSERT"

        elif isinstance(statement,exp.Update):
            operation="UPDATE"

        elif isinstance(statement,exp.Delete):
            operation="DELETE"

        elif isinstance(statement,exp.Alter):
            operation="ALTER"

        elif isinstance(statement,exp.Drop):
            operation="ALTER"

        elif isinstance(statement,exp.Create):
            operation="CREATE"

        else:
            return {
                "valid":False,
                "operation":None,
                "message":"Unsupported SQL operation"
            }

        return {
            "valid":True,
            "operation":operation,
            "message":f"{operation} query validated"
        }
    except Exception as e:
        return {
            "valid":False,
            "operation":None,
            "message":str(e)
        }


In [19]:
print(validate_sql("SELECT * FROM Track"))
print(validate_sql("DROP TABLE Track"))
print(validate_sql("DELETE FROM Track"))
print(validate_sql("UPDATE Track SET Name = 'Test'"))

{'valid': True, 'operation': 'SELECT', 'message': 'SELECT query validated'}
{'valid': True, 'operation': 'ALTER', 'message': 'ALTER query validated'}
{'valid': True, 'operation': 'DELETE', 'message': 'DELETE query validated'}
{'valid': True, 'operation': 'UPDATE', 'message': 'UPDATE query validated'}


### SQL Approval

In [20]:
def check_approval(operation):

    if operation=="SELECT":
        return False

    return True

In [21]:
operations=["DELETE","SELECT","INSERT","ALTER","UPDATE"]

for operation in operations:

    approval=check_approval(operation)

    if approval:
        print(operation,"-> Human approval required")

    else:
        print(operation,"->No approval required")

DELETE -> Human approval required
SELECT ->No approval required
INSERT -> Human approval required
ALTER -> Human approval required
UPDATE -> Human approval required


In [22]:
def prepare_sql(query):

    validation=validate_sql(query)

    if not validation['valid']:
        return {
            "status":"rejected",
            "query":query,
            "message":validation['message']
        }

    operation=validation['operation']

    approval_required=check_approval(operation)

    if approval_required:
        return {
            "status":"approval_required",
            "query":query,
            "operation":operation,
            "message":"Human Approval Required"
        }

    return {
        "status":"ready",
        "query":query,
        "operation":operation,
        "message":"Query is ready for execution"
    }


In [23]:
print(prepare_sql("SELECT * FROM Track LIMIT 5"))

print(prepare_sql(
    "UPDATE Artist SET Name = 'Updated Artist' WHERE ArtistId = 1"
))

print(prepare_sql("DROP TABLE Track"))

{'status': 'ready', 'query': 'SELECT * FROM Track LIMIT 5', 'operation': 'SELECT', 'message': 'Query is ready for execution'}
{'status': 'approval_required', 'query': "UPDATE Artist SET Name = 'Updated Artist' WHERE ArtistId = 1", 'operation': 'UPDATE', 'message': 'Human Approval Required'}
{'status': 'approval_required', 'query': 'DROP TABLE Track', 'operation': 'ALTER', 'message': 'Human Approval Required'}


### Agent

In [24]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Optional

In [25]:
class AgentState(TypedDict):

    question: str
    schema: str
    sql_query: str
    operation: Optional[str]
    status: str
    approval_required: bool
    result: Optional[str]
    answer:Optional[str]
    error:Optional[str]
    retry_count:int
    max_retries:int

### Response Agent

In [26]:
def response_chain(llm):

    prompt=ChatPromptTemplate.from_messages([
        ("system","""

        You are a helpful database assistant.
        Answer the user's question using only the SQL query results.
        Keep the answer simple and clear.
        If the results are empty, tell the user that no records were found.
        Do not make up any information.
        """),
        ("human", """
        User Question: {question}

        SQL Query: {sql_query}

        SQL Results: {result}

        Provide a natural language answer:


        """)
    ])

    return prompt|llm|StrOutputParser()

answer_chain=response_chain(llm)

In [27]:
def response_node(state:AgentState):

    answer=answer_chain.invoke({
        "question":state["question"],
        "sql_query":state["sql_query"],
        "result":state["result"]
    })

    return {
        "answer":answer
    }

In [28]:
# SQL Generator Node

def generate_sql_node(state:AgentState):

    response=sql_chain.invoke({
        "schema":state["schema"],
        "question":state["question"]
    })

    return {
        "sql_query":response
    }

In [29]:
test_state = {
    "question":"Show me all tracks by AC/DC",
    "schema":get_schema(),
    "sql_query":"",
    "operation":None,
    "status":"started",
    "approval_required":False,
    "result":None
}

output=generate_sql_node(test_state)
test_state.update(output)
print(output)

{'sql_query': "SELECT t.`Name` AS TrackName, a.`Title` AS AlbumTitle, ar.`Name` AS ArtistName\nFROM `track` t\nJOIN `album` a ON t.`AlbumId` = a.`AlbumId`\nJOIN `artist` ar ON a.`ArtistId` = ar.`ArtistId`\nWHERE ar.`Name` = 'AC/DC'\nORDER BY t.`Name`;"}


In [30]:
print(test_state["sql_query"])

SELECT t.`Name` AS TrackName, a.`Title` AS AlbumTitle, ar.`Name` AS ArtistName
FROM `track` t
JOIN `album` a ON t.`AlbumId` = a.`AlbumId`
JOIN `artist` ar ON a.`ArtistId` = ar.`ArtistId`
WHERE ar.`Name` = 'AC/DC'
ORDER BY t.`Name`;


In [31]:
def validate_sql_node(state:AgentState):

    decision=prepare_sql(state["sql_query"])

    return {
        "operation":decision.get("operation"),
        "status":decision.get("status"),
        "approval_required":decision["status"]=="approval_required"
    }

In [32]:
validation_output=validate_sql_node(test_state)

print(validation_output)

{'operation': 'SELECT', 'status': 'ready', 'approval_required': False}


### Router Function

In [33]:
from langgraph.types import interrupt , Command
from langgraph.checkpoint.memory import MemorySaver

In [34]:
def route_after_validation(state:AgentState):

    if state["status"]=="rejected":
        return "reject"

    elif state["approval_required"]:
        return "approval"

    else:
        return "execute"

In [35]:
test_state.update(validation_output)

print(route_after_validation(test_state))

execute


In [36]:
def approval_node(state:AgentState):

    decision=interrupt({
        "message":"Human approval required",
        "operation":state["operation"],
        "sql_query":state["sql_query"]
    })

    if decision=="approve":
        return {
            "status":"approved"
        }

    else:
        return {
            "status":"rejected"
        }

In [37]:
def route_after_approval(state:AgentState):

    if state['status']=="approved":
        return "execute"

    else:
        return "reject"

In [38]:
def execute_node(state: AgentState):
    try:
        result = run_query(state["sql_query"])

        return {
            "result": str(result),
            "status": "completed",
            "error": None
        }

    except Exception as e:
        return {
            "result": None,
            "status": "error",
            "error": str(e)
        }



def reject_node(state:AgentState):
    return {
        "status":"rejected"
    }

In [39]:
def correct_sql_query(llm):

    prompt=ChatPromptTemplate([
        ("system", """
        You are an expert MySQL query correction assistant.

        Your task is to fix SQL queries that failed during execution.

        Rules:
        - Analyze the SQL query and database error.
        - Generate corrected MySQL SQL.
        - Use only tables and columns present in the schema.
        - Do not change the user's original intention.
        - Return only the corrected SQL query.
        - Do not include explanations or markdown.
        """),

        ("human", """
        Database Schema:
        {schema}

        User Question:
        {question}

        Failed SQL Query:
        {sql_query}

        Database Error:
        {error}

        Corrected SQL Query:
        """)
    ])

    return prompt|llm|StrOutputParser()

correction_chain=correct_sql_query(llm)

In [40]:
def correction_node(state:AgentState):

    corrected_sql=correction_chain.invoke({
        "schema":state["schema"],
        "question":state["question"],
        "sql_query":state["sql_query"],
        "error":state["error"]
    })

    return {
        "sql_query":corrected_sql.strip(),
        "retry_count":state["retry_count"]+1,
        "status":"retry"
    }

In [41]:
def route_after_execution(state:AgentState):

    if state["status"]=="completed":
        return "response"

    elif state["status"]=="error":
        if state["retry_count"]<state['max_retries']:
            return "correct"
        else:
            return "reject"

    return "reject"

### Graph

In [42]:
graph=StateGraph(AgentState)

In [43]:
graph.add_node("generate_sql",generate_sql_node)
graph.add_node("validate_sql",validate_sql_node)
graph.add_node("execute",execute_node)
graph.add_node("approval",approval_node)
graph.add_node("reject",reject_node)
graph.add_node("response",response_node)
graph.add_node("correct",correction_node)

In [44]:
graph.add_edge(START,"generate_sql")

graph.add_edge("generate_sql","validate_sql")

# Validation Routing
graph.add_conditional_edges(
    "validate_sql",
    route_after_validation,
    {
        "execute":"execute",
        "approval":"approval",
        "reject":"reject"
    }
)

# Approval Routing
graph.add_conditional_edges(
    "approval",
    route_after_approval,
    {
        "execute":"execute",
        "reject":"reject"
    }
)

# Execution Routing
graph.add_conditional_edges(
    "execute",
    route_after_execution,
    {
        "response":"response",
        "correct":"correct",
        "reject":"reject"
    }
)

graph.add_edge("correct","validate_sql")

graph.add_edge("response",END)
graph.add_edge("reject",END)

In [45]:
memory=MemorySaver()

In [46]:
app=graph.compile(checkpointer=memory)

In [47]:
# Testing 

config={
    "configurable":{
        "thread_id":"querymind-hitl-test"
    }
}

result=app.invoke({
    "question":"Change the name of the artist with ArtistID 1 to AC/DC.",
    "schema":get_schema(),
    "sql_query":"",
    "operation":None,
    "status":"started",
    "approval_required":False,
    "result":None
    },config=config)

print(result)

{'question': 'Change the name of the artist with ArtistID 1 to AC/DC.', 'schema': '\nCREATE TABLE album (\n\t`AlbumId` BIGINT NOT NULL AUTO_INCREMENT, \n\t`Title` VARCHAR(160) NOT NULL, \n\t`ArtistId` BIGINT NOT NULL, \n\tPRIMARY KEY (`AlbumId`), \n\tCONSTRAINT album_ibfk_1 FOREIGN KEY(`ArtistId`) REFERENCES artist (`ArtistId`)\n)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci\n\n\nCREATE TABLE artist (\n\t`ArtistId` BIGINT NOT NULL AUTO_INCREMENT, \n\t`Name` VARCHAR(120), \n\tPRIMARY KEY (`ArtistId`)\n)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci\n\n\nCREATE TABLE customer (\n\t`CustomerId` BIGINT NOT NULL AUTO_INCREMENT, \n\t`FirstName` VARCHAR(40) NOT NULL, \n\t`LastName` VARCHAR(20) NOT NULL, \n\t`Company` VARCHAR(80), \n\t`Address` VARCHAR(70), \n\t`City` VARCHAR(40), \n\t`State` VARCHAR(40), \n\t`Country` VARCHAR(40), \n\t`PostalCode` VARCHAR(10), \n\t`Phone` VARCHAR(24), \n\t`Fax` VARCHAR(24), \n\t`Email` VARCHAR(60) NOT NULL, \n\t`SupportRep

In [48]:
result=app.invoke(
    Command(resume="approve"),
    config=config
)

print(result)

Query being executed:UPDATE artist SET Name = 'AC/DC' WHERE ArtistId = 1;
{'question': 'Change the name of the artist with ArtistID 1 to AC/DC.', 'schema': '\nCREATE TABLE album (\n\t`AlbumId` BIGINT NOT NULL AUTO_INCREMENT, \n\t`Title` VARCHAR(160) NOT NULL, \n\t`ArtistId` BIGINT NOT NULL, \n\tPRIMARY KEY (`AlbumId`), \n\tCONSTRAINT album_ibfk_1 FOREIGN KEY(`ArtistId`) REFERENCES artist (`ArtistId`)\n)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci\n\n\nCREATE TABLE artist (\n\t`ArtistId` BIGINT NOT NULL AUTO_INCREMENT, \n\t`Name` VARCHAR(120), \n\tPRIMARY KEY (`ArtistId`)\n)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci\n\n\nCREATE TABLE customer (\n\t`CustomerId` BIGINT NOT NULL AUTO_INCREMENT, \n\t`FirstName` VARCHAR(40) NOT NULL, \n\t`LastName` VARCHAR(20) NOT NULL, \n\t`Company` VARCHAR(80), \n\t`Address` VARCHAR(70), \n\t`City` VARCHAR(40), \n\t`State` VARCHAR(40), \n\t`Country` VARCHAR(40), \n\t`PostalCode` VARCHAR(10), \n\t`Phone` VARCHAR(24)

In [49]:
db.run("""
SELECT ArtistId, Name
FROM artist
WHERE ArtistId = 1;
""")

"[(1, 'AC/DC')]"

In [50]:
config={
    "configurable":{
        "thread_id":"querymind-select-test"
    }
}

In [51]:
result = app.invoke({
    "question": "Show all tracks by AC/DC",
    "schema": get_schema(),
    "sql_query": "",
    "operation": None,
    "status": "started",
    "approval_required": False,
    "result": None,
    "answer": None
}, config=config)

Query being executed:SELECT t.`Name` AS TrackName, a.`Title` AS AlbumTitle
FROM `track` t
JOIN `album` a ON t.`AlbumId` = a.`AlbumId`
JOIN `artist` ar ON a.`ArtistId` = ar.`ArtistId`
WHERE ar.`Name` = 'AC/DC';


In [52]:
print("Generated SQL:\n", result["sql_query"])
print("\nDatabase Result:\n", result["result"])
print("\nFinal Answer:\n", result["answer"])

Generated SQL:
 SELECT t.`Name` AS TrackName, a.`Title` AS AlbumTitle
FROM `track` t
JOIN `album` a ON t.`AlbumId` = a.`AlbumId`
JOIN `artist` ar ON a.`ArtistId` = ar.`ArtistId`
WHERE ar.`Name` = 'AC/DC';

Database Result:
 [('For Those About To Rock (We Salute You)', 'For Those About To Rock We Salute You'), ('Put The Finger On You', 'For Those About To Rock We Salute You'), ("Let's Get It Up", 'For Those About To Rock We Salute You'), ('Inject The Venom', 'For Those About To Rock We Salute You'), ('Snowballed', 'For Those About To Rock We Salute You'), ('Evil Walks', 'For Those About To Rock We Salute You'), ('C.O.D.', 'For Those About To Rock We Salute You'), ('Breaking The Rules', 'For Those About To Rock We Salute You'), ('Night Of The Long Knives', 'For Those About To Rock We Salute You'), ('Spellbound', 'For Those About To Rock We Salute You'), ('Go Down', 'Let There Be Rock'), ('Dog Eat Dog', 'Let There Be Rock'), ('Let There Be Rock', 'Let There Be Rock'), ('Bad Boy Boogie', '

In [53]:
test_state = {
    "question": "Show all tracks by AC/DC",
    "schema": get_schema(),
    "sql_query": "SELECT WrongColumn FROM artist;",
    "operation": "SELECT",
    "status": "ready",
    "approval_required": False,
    "result": None,
    "answer": None,
    "error": None,
    "retry_count": 0,
    "max_retries": 2
}

output = execute_node(test_state)
print(output)

Query being executed:SELECT WrongColumn FROM artist;
{'result': None, 'status': 'error', 'error': '(pymysql.err.OperationalError) (1054, "Unknown column \'WrongColumn\' in \'field list\'")\n[SQL: SELECT WrongColumn FROM artist;]\n(Background on this error at: https://sqlalche.me/e/21/e3q8)'}


In [54]:
correction_output = correction_node({
    **test_state,
    "error": output["error"]
})

print(correction_output)

{'sql_query': "SELECT t.* \nFROM track t \nJOIN album a ON t.AlbumId = a.AlbumId \nJOIN artist ar ON a.ArtistId = ar.ArtistId \nWHERE ar.Name = 'AC/DC';", 'retry_count': 1, 'status': 'retry'}


In [55]:
validation_output = prepare_sql(correction_output["sql_query"])

print(validation_output)

{'status': 'ready', 'query': "SELECT t.* \nFROM track t \nJOIN album a ON t.AlbumId = a.AlbumId \nJOIN artist ar ON a.ArtistId = ar.ArtistId \nWHERE ar.Name = 'AC/DC';", 'operation': 'SELECT', 'message': 'Query is ready for execution'}


In [56]:
test_state = {
    "question": "Show all tracks by AC/DC",
    "schema": get_schema(),
    "sql_query": "SELECT WrongColumn FROM artist;",
    "operation": "SELECT",
    "status": "ready",
    "approval_required": False,
    "result": None,
    "answer": None,
    "error": None,
    "retry_count": 0,
    "max_retries": 2
}

config = {
    "configurable": {
        "thread_id": "querymind-retry-test"
    }
}

result = app.invoke(test_state, config=config)

print("Status:", result["status"])
print("Retry Count:", result["retry_count"])
print("Corrected SQL:", result["sql_query"])
print("Answer:", result["answer"])

Query being executed:SELECT t.TrackId, t.Name AS TrackName, a.Title AS AlbumTitle
FROM track t
JOIN album a ON t.AlbumId = a.AlbumId
JOIN artist ar ON a.ArtistId = ar.ArtistId
WHERE ar.Name = 'AC/DC';
Status: completed
Retry Count: 0
Corrected SQL: SELECT t.TrackId, t.Name AS TrackName, a.Title AS AlbumTitle
FROM track t
JOIN album a ON t.AlbumId = a.AlbumId
JOIN artist ar ON a.ArtistId = ar.ArtistId
WHERE ar.Name = 'AC/DC';
Answer: Here are all the tracks by AC/DC:

- **For Those About To Rock (We Salute You)**
  - For Those About To Rock (We Salute You)
  - Put The Finger On You
  - Let's Get It Up
  - Inject The Venom
  - Snowballed
  - Evil Walks
  - C.O.D.
  - Breaking The Rules
  - Night Of The Long Knives
  - Spellbound

- **Let There Be Rock**
  - Go Down
  - Dog Eat Dog
  - Let There Be Rock
  - Bad Boy Boogie
  - Problem Child
  - Overdose
  - Hell Ain't A Bad Place To Be
  - Whole Lotta Rosie


In [57]:
retry_graph = StateGraph(AgentState)

retry_graph.add_node("validate_sql", validate_sql_node)
retry_graph.add_node("execute", execute_node)
retry_graph.add_node("correct", correction_node)
retry_graph.add_node("approval", approval_node)
retry_graph.add_node("reject", reject_node)
retry_graph.add_node("response", response_node)

retry_graph.add_edge(START, "validate_sql")

retry_graph.add_conditional_edges(
    "validate_sql",
    route_after_validation,
    {
        "execute": "execute",
        "approval": "approval",
        "reject": "reject"
    }
)

retry_graph.add_conditional_edges(
    "execute",
    route_after_execution,
    {
        "response": "response",
        "correct": "correct",
        "reject": "reject"
    }
)

retry_graph.add_edge("correct", "validate_sql")

retry_graph.add_conditional_edges(
    "approval",
    route_after_approval,
    {
        "execute": "execute",
        "reject": "reject"
    }
)

retry_graph.add_edge("response", END)
retry_graph.add_edge("reject", END)

retry_app = retry_graph.compile(checkpointer=memory)

In [58]:
retry_test_state = {
    "question": "Show all tracks by AC/DC",
    "schema": get_schema(),
    "sql_query": "SELECT WrongColumn FROM artist;",
    "operation": None,
    "status": "ready",
    "approval_required": False,
    "result": None,
    "answer": None,
    "error": None,
    "retry_count": 0,
    "max_retries": 2
}

retry_config = {
    "configurable": {
        "thread_id": "querymind-retry-test-2"
    }
}

retry_result = retry_app.invoke(
    retry_test_state,
    config=retry_config
)

print("Status:", retry_result["status"])
print("Retry Count:", retry_result["retry_count"])
print("Final SQL:", retry_result["sql_query"])
print("Answer:", retry_result["answer"])

Query being executed:SELECT WrongColumn FROM artist;
Query being executed:SELECT t.* 
FROM track t 
JOIN album a ON t.AlbumId = a.AlbumId 
JOIN artist ar ON a.ArtistId = ar.ArtistId 
WHERE ar.Name = 'AC/DC';
Status: completed
Retry Count: 1
Final SQL: SELECT t.* 
FROM track t 
JOIN album a ON t.AlbumId = a.AlbumId 
JOIN artist ar ON a.ArtistId = ar.ArtistId 
WHERE ar.Name = 'AC/DC';
Answer: Here are all the tracks by AC/DC:

- For Those About To Rock (We Salute You)  
- Put The Finger On You  
- Let's Get It Up  
- Inject The Venom  
- Snowballed  
- Evil Walks  
- C.O.D.  
- Breaking The Rules  
- Night Of The Long Knives  
- Spellbound  
- Go Down  
- Dog Eat Dog  
- Let There Be Rock  
- Bad Boy Boogie  
- Problem Child  
- Overdose  
- Hell Ain't A Bad Place To Be  
- Whole Lotta Rosie


In [59]:
from fastmcp import FastMCP 

mcp=FastMCP("QueryMind")

print("QueryMind MCP Server Initialized")

QueryMind MCP Server Initialized


In [60]:
@mcp.tool()
def get_tables() -> list:
    """Get the list of all available tables in the connected database."""
    return db.get_usable_table_names()

In [61]:
get_tables()

['album',
 'artist',
 'customer',
 'employee',
 'genre',
 'invoice',
 'invoiceline',
 'mediatype',
 'playlist',
 'playlisttrack',
 'track']

In [62]:
@mcp.tool()
def get_schema() -> str:
    """Get the complete schema of the connected database."""
    return db.get_table_info()

In [63]:
schema=get_schema()
print(schema[:2000])


CREATE TABLE album (
	`AlbumId` BIGINT NOT NULL AUTO_INCREMENT, 
	`Title` VARCHAR(160) NOT NULL, 
	`ArtistId` BIGINT NOT NULL, 
	PRIMARY KEY (`AlbumId`), 
	CONSTRAINT album_ibfk_1 FOREIGN KEY(`ArtistId`) REFERENCES artist (`ArtistId`)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci


CREATE TABLE artist (
	`ArtistId` BIGINT NOT NULL AUTO_INCREMENT, 
	`Name` VARCHAR(120), 
	PRIMARY KEY (`ArtistId`)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci


CREATE TABLE customer (
	`CustomerId` BIGINT NOT NULL AUTO_INCREMENT, 
	`FirstName` VARCHAR(40) NOT NULL, 
	`LastName` VARCHAR(20) NOT NULL, 
	`Company` VARCHAR(80), 
	`Address` VARCHAR(70), 
	`City` VARCHAR(40), 
	`State` VARCHAR(40), 
	`Country` VARCHAR(40), 
	`PostalCode` VARCHAR(10), 
	`Phone` VARCHAR(24), 
	`Fax` VARCHAR(24), 
	`Email` VARCHAR(60) NOT NULL, 
	`SupportRepId` BIGINT, 
	PRIMARY KEY (`CustomerId`), 
	CONSTRAINT customer_ibfk_1 FOREIGN KEY(`SupportRepId`) REFERENCES employee (`EmployeeId`)


In [64]:
@mcp.tool()
def generate_sql(question:str)->str:
    """Generate a SQL Query based on the user's natural language question"""

    schema=get_schema()

    query=sql_chain.invoke({
        "schema":schema,
        "question":question
    })

    return query.strip()

In [65]:
generate_sql("Show me all tracks by AC/DC.")

"SELECT t.TrackId, t.Name AS TrackName, a.Title AS AlbumTitle\nFROM track t\nJOIN album a ON t.AlbumId = a.AlbumId\nJOIN artist ar ON a.ArtistId = ar.ArtistId\nWHERE ar.Name = 'AC/DC';"

In [66]:
@mcp.tool()
def validate_sql_query(query:str)->str:
    """Validate SQL Syntax and identify the SQL Operation"""
    return validate_sql(query)

In [67]:
validate_sql_query(
    "SELECT * FROM artist;"
    )

{'valid': True, 'operation': 'SELECT', 'message': 'SELECT query validated'}

In [68]:
validate_sql_query(
    "SELECT WrongColumn FROM artist;"
)

{'valid': True, 'operation': 'SELECT', 'message': 'SELECT query validated'}

In [69]:
@mcp.tool()
def execute_sql(query:str,approved:bool=False)->dict:
    """Execute a validated SQL Query.Write and DDL Operations require human approval"""

    validation=validate_sql(query)

    if not validation["valid"]:
        return {
            "status":"rejected",
            "message":validation["message"]
        }

    operation=validation["operation"]

    if check_approval(operation) and not approved:
        return {
            "status": "approval_required",
            "operation": operation,
            "query": query,
            "message": "Human approval required before execution."
        }

    try:
        result=run_query(query)

        return {
            "status":"completed",
            "result":str(result)
        }

    except Exception as e:
        return {
            "status":"error",
            "error":str(e)
        }

In [70]:
execute_sql("SELECT * FROM artist LIMIT 5;")

Query being executed:SELECT * FROM artist LIMIT 5;


{'status': 'completed',
 'result': "[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]"}

In [71]:
execute_sql("UPDATE artist SET Name = 'Test' WHERE ArtistId = 1;")

{'status': 'approval_required',
 'operation': 'UPDATE',
 'query': "UPDATE artist SET Name = 'Test' WHERE ArtistId = 1;",
 'message': 'Human approval required before execution.'}

In [72]:
@mcp.tool()
def get_sample_data(table_name:str)->str:
    """Retrieve sample records from a database table"""

    allowed_tables=db.get_usable_table_names()

    if table_name not in allowed_tables:
        return "Error:Table does not exist"

    query=f"SELECT * FROM {table_name} LIMIT 5;"

    return run_query(query)

In [73]:
get_sample_data("artist")

Query being executed:SELECT * FROM artist LIMIT 5;


"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]"

In [74]:
import uuid

@mcp.tool()
def ask_database(question:str)->dict:
    """Ask a natural-language question about the database.
    Use the Langgraph SQL Agent to generate, validate ,
    execute and explain SQL Queries"""

    thread_id=str(uuid.uuid4())

    initial_state = {
        "question":question,
        "schema":get_schema(),
        "sql_query":"",
        "operation":None,
        "status":"pending",
        "approval_required":False,
        "result":None,
        "answer":None,
        "error":None,
        "retry_count":0,
        "max_retries":2
    }

    config={
        "configurable":{
            "thread_id":thread_id
        }
    }

    result=app.invoke(
        initial_state,
        config=config
    )

    if result.get("__interrupt__"):
        return {
            "status":"approval_required",
            "thread_id":thread_id,
            "approval_request":result["__interrupt__"][0].value
        }

    return {
        "status":result['status'],
        "sql_query":result["sql_query"],
        "answer":result["answer"],
        "retry_count":result["retry_count"]
    }

In [75]:
ask_database("show all tracks by AC/DC")

Query being executed:SELECT t.`Name` AS TrackName, a.`Title` AS AlbumTitle, ar.`Name` AS ArtistName
FROM `track` t
JOIN `album` a ON t.`AlbumId` = a.`AlbumId`
JOIN `artist` ar ON a.`ArtistId` = ar.`ArtistId`
WHERE ar.`Name` = 'AC/DC';


{'status': 'completed',
 'sql_query': "SELECT t.`Name` AS TrackName, a.`Title` AS AlbumTitle, ar.`Name` AS ArtistName\nFROM `track` t\nJOIN `album` a ON t.`AlbumId` = a.`AlbumId`\nJOIN `artist` ar ON a.`ArtistId` = ar.`ArtistId`\nWHERE ar.`Name` = 'AC/DC';",
 'answer': 'Here are all the tracks by AC/DC that appear in the database:\n\n**Album: “For Those About To Rock We Salute You”**\n- For Those About To Rock (We Salute You)\n- Put The Finger On You\n- Let’s Get It Up\n- Inject The Venom\n- Snowballed\n- Evil Walks\n- C.O.D.\n- Breaking The Rules\n- Night Of The Long Knives\n- Spellbound\n\n**Album: “Let There Be Rock”**\n- Go Down\n- Dog Eat Dog\n- Let There Be Rock\n- Bad Boy Boogie\n- Problem Child\n- Overdose\n- Hell Ain’t A Bad Place To Be\n- Whole Lotta Rosie',
 'retry_count': 0}

In [78]:
from langgraph.types import Command 

@mcp.tool()
def approve_database(thread_id:str,decision:str)->dict:
    """Approve or Reject a pending SQL Operation and resume the workflow"""

    if decision not in ["approve","reject"]:
        return {
            "status":"error",
            "message":"Decision must be 'approve' or 'reject'"
        }

    config = {
        "configurable":{
            "thread_id":thread_id
        }
    }

    try:
        result=app.invoke(
            Command(resume=decision),
            config=config
        )

        return {
            "status":result['status'],
            "sql_query":result.get("sql_query"),
            "answer":result.get("answer"),
            "message":"Workflow resumed successfully"
        }

    except Exception as e:
        return {
            "status":"error",
            "message":str(e)
        }

[11/22/26 22:55:58] WARNING  Component already exists: tool:approve_database@                 ]8;id=6147326;file://c:\QueryMind\QueryMind\venv\Lib\site-packages\fastmcp\server\providers\local_provider\local_provider.py\local_provider.py]8;;\:]8;id=6147327;file://c:\QueryMind\QueryMind\venv\Lib\site-packages\fastmcp\server\providers\local_provider\local_provider.py#192\192]8;;\

In [79]:
approval_test = ask_database(
    "Change the name of artist with ArtistId 1 to Test Artist"
)

print(approval_test)

{'status': 'approval_required', 'thread_id': '91ea5fa8-bb5a-49bb-8009-b0b56e138945', 'approval_request': {'message': 'Human approval required', 'operation': 'UPDATE', 'sql_query': "UPDATE artist SET Name = 'Test Artist' WHERE ArtistId = 1;"}}
